# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [14]:
import sys
from pathlib import Path
import gradio as gr

# Add the parent folder (root) to Python's module search path
sys.path.append(str(Path.cwd().parent))

In [15]:

# Now Python can find router.py
from router import PROVIDERS, API_KEYS, MODEL_CONFIG
from openai import OpenAI

client = OpenAI(base_url=PROVIDERS["OPENROUTER"], api_key=API_KEYS["OPENROUTER"])

In [16]:
MODEL_CONFIG['openrouter']['models']['dots']

'dots-studio/dots-3-note-preview:free'

## Test model

In [17]:
res = client.chat.completions.create(
    messages=[{"role": "user", "content": "Hi!"}],
    model=MODEL_CONFIG['openrouter']['models']['dots']
)

res.choices[0].message.content

"\n\nHi! I'm dots, an AI model developed by Dots Studio at rednote. How can I help you today?"

In [37]:
SYSTEM_MESSAGE = "You are a great joker. You start and end with relevant emojis"

In [6]:
AVAILABLE_PROVIDERS = list(PROVIDERS.keys())
AVAILABLE_PROVIDERS

['OPENROUTER', 'GROQ', 'POLLINATIONS']

In [7]:
def speak(message):
    speaker_client = OpenAI(base_url=PROVIDERS['OPENROUTER'], api_key=API_KEYS['OPENROUTER'])
    response = speaker_client.audio.speech.create(
        model="deepgram/flux-tts:free",
        voice="flux-bree-en",
        input=message,
    )
    return response.content

In [8]:
import tempfile

In [9]:
def chat(provider, model, message):
    chat_client = OpenAI(base_url=PROVIDERS[provider],api_key=API_KEYS[provider])
    audio_client = OpenAI(base_url=PROVIDERS['OPENROUTER'],api_key=API_KEYS['OPENROUTER'])
    
    chat_response = chat_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": message}
        ]
    )

    chat_text = chat_response.choices[0].message.content
    yield chat_text, None
    
    # Get the audiobytes as output with the obtained chat_repsonse
    response = audio_client.audio.speech.create(
        model="deepgram/flux-tts:free",
        voice="flux-bruce-en",
        input=chat_text,
    )
        
    with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as tmp_file:
        tmp_file.write(response.content)
        audio_path = tmp_file.name

    return chat_text, audio_path

In [35]:
def chat_with_history(provider, model, message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": SYSTEM_MESSAGE}]
        + history
        + [{"role": "user", "content": message}]
    )

    chat_client = OpenAI(
        base_url=PROVIDERS[provider],
        api_key=API_KEYS[provider],
    )

    response = chat_client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True,
    )

    chat_history = [
        {"role": "user", "content": message},
        {"role": "assistant", "content": ""},
    ]

    answer = ""
    for chunk in response:
        content = chunk.choices[0].delta.content
        if content:
            answer += content
            chat_history[-1]["content"] = answer

            yield chat_history, None, message

    audio_response = audio_client.audio.speech.create(
        model="deepgram/flux-tts:free",
        voice="flux-bruce-en",
        input=answer,
    )

    with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as tmp_file:
        tmp_file.write(audio_response.content)
        audio_path = tmp_file.name

    yield chat_history, audio_path, ""

In [10]:
def get_models(provider):
    return list(MODEL_CONFIG[provider.lower()]['models'].values())

In [11]:

def update_models(provider):
    models = get_models(provider)
    return gr.Dropdown(choices=models, value=models[0])


In [38]:
with gr.Blocks(title="Question Answer Machine") as demo:
    gr.Markdown("# Question Answer Machine")

    with gr.Row():
        provider_dd = gr.Dropdown(
            choices=AVAILABLE_PROVIDERS,
            value=AVAILABLE_PROVIDERS[0],
            label="Select a Provider",
        )
        model_dd = gr.Dropdown(
            choices=get_models(AVAILABLE_PROVIDERS[0]),
            value=get_models(AVAILABLE_PROVIDERS[0])[0],
            label="Select a model",
        )

    chatbot = gr.Chatbot(label="Conversation", height=200)
    prompt = gr.Textbox(
        label="Enter your prompt",
        placeholder="Ask a question...",
        lines=2,
    )
    submit_btn = gr.Button("Submit", variant="primary")
    clear_btn = gr.Button("Clear")

    output_audio = gr.Audio(
        label="Generated Speech",
        type="filepath",
        autoplay=True,
    )

    provider_dd.change(
        update_models,
        inputs=provider_dd,
        outputs=model_dd,
    )

    submit_btn.click(
        chat_with_history,
        inputs=[provider_dd, model_dd, prompt, chatbot],
        outputs=[chatbot, output_audio, prompt],
    )

    prompt.submit(
        chat_with_history,
        inputs=[provider_dd, model_dd, prompt, chatbot],
        outputs=[chatbot, output_audio, prompt],
    )

    clear_btn.click(
        lambda: ([], None),
        outputs=[chatbot, output_audio],
    )

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


In [13]:
audio_client = OpenAI(
    base_url=PROVIDERS["OPENROUTER"],
    api_key=API_KEYS['OPENROUTER']
)

resp = audio_client.audio.speech.with_streaming_response.create(
    model="deepgram/flux-tts:free",
    voice="flux-bree-en",
    input="test",
)
print(type(resp))

<class 'openai._response.ResponseContextManager'>
